# 02 - Qualitative external inference on Houston airport imagery

This notebook applies the final CORS-ADD-HBB model to independently sourced Houston airport imagery. It tiles large remote-sensing images, runs YOLOv5x inference, maps detections back to original image coordinates, removes duplicates in overlapping tiles, and saves full-image visualizations.

Without complete Houston ground-truth annotations, these results are qualitative and must not be reported as Precision, Recall, F1, or mAP.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import csv
import json
import math
import re
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.ops import nms

PROJECT_ROOT = Path('/content/drive/MyDrive/RS_PRO')
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
YOLOV5_ROOT = Path('/content/yolov5')

FINAL_WEIGHTS = (
    ARTIFACT_ROOT
    / 'runs'
    / 'stage2_cors_add_hbb_yolov5x'
    / 'weights'
    / 'best.pt'
)

HOUSTON_INPUT = PROJECT_ROOT / 'Houston_Imagery' / 'input'
HOUSTON_OUTPUT = ARTIFACT_ROOT / 'houston_external_inference'

LOCAL_TILE_ROOT = Path('/content/houston_tiles')
LOCAL_DETECTION_ROOT = Path('/content/houston_detection')

TILE_SIZE = 640
TILE_OVERLAP = 0.25
CONFIDENCE_THRESHOLD = 0.25
DETECTION_IOU_THRESHOLD = 0.45
GLOBAL_NMS_IOU_THRESHOLD = 0.40

HOUSTON_INPUT.mkdir(parents=True, exist_ok=True)
HOUSTON_OUTPUT.mkdir(parents=True, exist_ok=True)

print('Place independently sourced Houston images in:', HOUSTON_INPUT)
print('Final weights:', FINAL_WEIGHTS)


## Record image provenance

Before inference, create or complete `Houston_Imagery/metadata.csv` with one row per source image. Record the provider, acquisition date, spatial resolution, airport, and confirmation that the image does not duplicate or overlap CORS-ADD imagery.


In [ ]:
metadata_path = PROJECT_ROOT / 'Houston_Imagery' / 'metadata.csv'

if not metadata_path.exists():
    pd.DataFrame(
        columns=[
            'filename',
            'airport',
            'source_or_provider',
            'source_url',
            'acquisition_date',
            'spatial_resolution',
            'license_or_terms',
            'non_overlap_with_cors_add_confirmed',
            'notes',
        ]
    ).to_csv(metadata_path, index=False)
    print('Created metadata template:', metadata_path)
else:
    display(pd.read_csv(metadata_path))


## Install YOLOv5 if this is a new Colab runtime


In [ ]:
YOLOV5_REF = 'v7.0'

if not YOLOV5_ROOT.exists():
    subprocess.run(
        [
            'git', 'clone', '--depth', '1', '--branch', YOLOV5_REF,
            'https://github.com/ultralytics/yolov5.git',
            str(YOLOV5_ROOT),
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(YOLOV5_ROOT / 'requirements.txt')],
    check=True,
)

if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(
        f'Final Stage-2 weights are missing: {FINAL_WEIGHTS}. Run notebook 01 first.'
    )


## Tile the large Houston images


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}


def slugify(value: str) -> str:
    value = re.sub(r'[^A-Za-z0-9_-]+', '_', value)
    return value.strip('_') or 'image'


def axis_positions(length: int, tile_size: int, stride: int) -> list[int]:
    if length <= tile_size:
        return [0]
    positions = list(range(0, length - tile_size + 1, stride))
    final_position = length - tile_size
    if positions[-1] != final_position:
        positions.append(final_position)
    return positions


def tile_images(
    source_paths: list[Path],
    tile_root: Path,
    tile_size: int,
    overlap: float,
) -> pd.DataFrame:
    if tile_root.exists():
        shutil.rmtree(tile_root)
    tile_root.mkdir(parents=True, exist_ok=True)

    stride = max(1, round(tile_size * (1 - overlap)))
    manifest_rows = []

    for source_index, source_path in enumerate(source_paths):
        with Image.open(source_path) as source_image:
            source_image = source_image.convert('RGB')
            width, height = source_image.size

            x_positions = axis_positions(width, tile_size, stride)
            y_positions = axis_positions(height, tile_size, stride)

            for y0 in y_positions:
                for x0 in x_positions:
                    x1 = min(x0 + tile_size, width)
                    y1 = min(y0 + tile_size, height)
                    tile = source_image.crop((x0, y0, x1, y1))

                    tile_name = (
                        f'{source_index:03d}_{slugify(source_path.stem)}'
                        f'__x{x0}__y{y0}.png'
                    )
                    tile_path = tile_root / tile_name
                    tile.save(tile_path)

                    manifest_rows.append({
                        'tile_name': tile_name,
                        'tile_path': str(tile_path),
                        'source_path': str(source_path),
                        'source_name': source_path.name,
                        'x0': x0,
                        'y0': y0,
                        'tile_width': x1 - x0,
                        'tile_height': y1 - y0,
                        'source_width': width,
                        'source_height': height,
                    })

    return pd.DataFrame(manifest_rows)


houston_images = sorted(
    path for path in HOUSTON_INPUT.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not houston_images:
    raise FileNotFoundError(
        f'No Houston images found in {HOUSTON_INPUT}. Add images before continuing.'
    )

tile_manifest = tile_images(
    houston_images,
    LOCAL_TILE_ROOT,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
)

tile_manifest_path = HOUSTON_OUTPUT / 'tile_manifest.csv'
tile_manifest.to_csv(tile_manifest_path, index=False)

print(f'Source images: {len(houston_images)}')
print(f'Tiles: {len(tile_manifest)}')
print('Manifest:', tile_manifest_path)
display(tile_manifest.head())


## Run tile-level inference


In [ ]:
RUN_HOUSTON_INFERENCE = False  # Change to True after checking the source images and metadata.

detection_command = [
    sys.executable,
    str(YOLOV5_ROOT / 'detect.py'),
    '--weights', str(FINAL_WEIGHTS),
    '--source', str(LOCAL_TILE_ROOT),
    '--img', str(TILE_SIZE),
    '--conf-thres', str(CONFIDENCE_THRESHOLD),
    '--iou-thres', str(DETECTION_IOU_THRESHOLD),
    '--save-txt',
    '--save-conf',
    '--project', str(LOCAL_DETECTION_ROOT.parent),
    '--name', LOCAL_DETECTION_ROOT.name,
    '--exist-ok',
]

print(' '.join(detection_command))
if RUN_HOUSTON_INFERENCE:
    if LOCAL_DETECTION_ROOT.exists():
        shutil.rmtree(LOCAL_DETECTION_ROOT)
    subprocess.run(detection_command, cwd=YOLOV5_ROOT, check=True)
else:
    print('Houston inference was not started.')


## Map tile detections back to full images and remove duplicates


In [ ]:
def yolo_detection_to_global(
    parts: list[str],
    tile_row: pd.Series,
) -> dict:
    class_id, x_center, y_center, width, height, confidence = map(float, parts)

    tile_width = float(tile_row['tile_width'])
    tile_height = float(tile_row['tile_height'])

    x1 = (x_center - width / 2) * tile_width + float(tile_row['x0'])
    y1 = (y_center - height / 2) * tile_height + float(tile_row['y0'])
    x2 = (x_center + width / 2) * tile_width + float(tile_row['x0'])
    y2 = (y_center + height / 2) * tile_height + float(tile_row['y0'])

    return {
        'source_path': tile_row['source_path'],
        'source_name': tile_row['source_name'],
        'class_id': int(class_id),
        'confidence': confidence,
        'x1': max(0.0, x1),
        'y1': max(0.0, y1),
        'x2': min(float(tile_row['source_width']), x2),
        'y2': min(float(tile_row['source_height']), y2),
        'tile_name': tile_row['tile_name'],
    }


def collect_global_detections(
    manifest: pd.DataFrame,
    labels_root: Path,
) -> pd.DataFrame:
    columns = [
        'source_path', 'source_name', 'class_id', 'confidence',
        'x1', 'y1', 'x2', 'y2', 'tile_name',
    ]
    detections = []
    for _, row in manifest.iterrows():
        label_path = labels_root / f"{Path(row['tile_name']).stem}.txt"
        if not label_path.exists():
            continue

        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.split()
            if len(parts) != 6:
                continue
            detections.append(yolo_detection_to_global(parts, row))

    return pd.DataFrame(detections, columns=columns)


def apply_global_nms(
    detections: pd.DataFrame,
    iou_threshold: float,
) -> pd.DataFrame:
    if detections.empty:
        return detections.copy()

    retained_frames = []
    for source_path, frame in detections.groupby('source_path', sort=True):
        boxes = torch.tensor(
            frame[['x1', 'y1', 'x2', 'y2']].to_numpy(),
            dtype=torch.float32,
        )
        scores = torch.tensor(frame['confidence'].to_numpy(), dtype=torch.float32)
        keep_indices = nms(boxes, scores, iou_threshold).cpu().numpy()
        retained_frames.append(frame.iloc[keep_indices])

    return pd.concat(retained_frames, ignore_index=True)


labels_root = LOCAL_DETECTION_ROOT / 'labels'
if not labels_root.exists():
    raise FileNotFoundError(
        f'Detection labels not found: {labels_root}. Run the inference cell first.'
    )

raw_detections = collect_global_detections(tile_manifest, labels_root)
final_detections = apply_global_nms(
    raw_detections,
    iou_threshold=GLOBAL_NMS_IOU_THRESHOLD,
)

raw_detections.to_csv(HOUSTON_OUTPUT / 'raw_tile_detections.csv', index=False)
final_detections.to_csv(HOUSTON_OUTPUT / 'final_global_detections.csv', index=False)

print('Raw tile detections:', len(raw_detections))
print('Detections after global NMS:', len(final_detections))
display(final_detections.head())


## Save full-image detection visualizations


In [ ]:
visualization_root = HOUSTON_OUTPUT / 'visualizations'
visualization_root.mkdir(parents=True, exist_ok=True)

for source_path_string in sorted(final_detections['source_path'].unique()):
    source_path = Path(source_path_string)
    with Image.open(source_path) as image:
        image = image.convert('RGB')
        draw = ImageDraw.Draw(image)

        frame = final_detections[
            final_detections['source_path'] == source_path_string
        ].sort_values('confidence', ascending=False)

        line_width = max(2, min(image.size) // 1000)
        for _, detection in frame.iterrows():
            box = (
                detection['x1'],
                detection['y1'],
                detection['x2'],
                detection['y2'],
            )
            draw.rectangle(box, outline='red', width=line_width)
            label = f"aircraft {detection['confidence']:.2f}"
            draw.text((detection['x1'], max(0, detection['y1'] - 14)), label, fill='red')

        output_path = visualization_root / f'{source_path.stem}_detections.jpg'
        image.save(output_path, quality=95)
        print('Saved:', output_path)


## Preview results


In [ ]:
visualization_paths = sorted(visualization_root.glob('*.jpg'))

if not visualization_paths:
    print('No visualizations were produced.')
else:
    figure, axes = plt.subplots(
        len(visualization_paths),
        1,
        figsize=(16, 8 * len(visualization_paths)),
    )
    if len(visualization_paths) == 1:
        axes = [axes]

    for axis, visualization_path in zip(axes, visualization_paths):
        image = Image.open(visualization_path)
        axis.imshow(image)
        axis.set_title(visualization_path.name)
        axis.axis('off')

    plt.tight_layout()
    plt.show()


## Interpretation rule

Report these Houston results as qualitative external inference unless complete ground-truth annotations are created. Discuss representative correct detections, apparent false positives, apparent false negatives, dense parking areas, small aircraft, and background complexity. Do not report mAP, Precision, Recall, or F1 from unannotated Houston imagery.
